# 🚀 STAGE 3 보스전 — Colab GPU 버전

> DONGA KDT 허깅페이스 과정 / 3일차 파인튜닝

## 왜 Colab인가

내 노트북 CPU에서는 1,500문장 × 2에폭에 **8분**이 걸렸고, 그마저도 학습이 부족했습니다.

| | CPU (내 노트북) | Colab GPU (T4) |
|---|---|---|
| 문장 수 | 1,500 | **20,000** |
| 예상 시간 | 8분 | **약 6~10분** |
| 결과 | 검증 70% | 검증 88~90% 기대 |

**13배 많은 데이터를 비슷한 시간에** 학습합니다. 이게 GPU를 쓰는 이유입니다.

---

## ⚠️ 시작 전 필수 설정

1. 상단 메뉴 **런타임 → 런타임 유형 변경**
2. 하드웨어 가속기 → **T4 GPU** 선택 → 저장
3. 아래 셀을 위에서부터 차례로 실행 (`Shift + Enter`)

설정을 안 하면 CPU로 돌아가서 몇 시간 걸립니다. 꼭 확인하세요.


## 1️⃣ GPU 확인

가장 먼저 할 일. GPU가 안 잡혔으면 여기서 멈추고 런타임 설정을 다시 하세요.


In [ ]:
import torch

if torch.cuda.is_available():
    print('✅ GPU 사용 가능:', torch.cuda.get_device_name(0))
    print('   메모리:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('❌ GPU가 없습니다!')
    print('   런타임 → 런타임 유형 변경 → T4 GPU 선택 후 다시 실행하세요.')
    raise SystemExit('GPU 설정 필요')


## 2️⃣ 패키지 설치

Colab에는 이미 torch가 깔려 있습니다. 나머지만 설치합니다.
설치 후 런타임 재시작 안내가 뜨면 무시하고 계속 진행하세요.


In [ ]:
!pip install -q transformers==4.44.2 accelerate==0.34.0 datasets==3.2.0 evaluate scikit-learn

import transformers, datasets
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)


## 3️⃣ 설정

### 왜 `klue/roberta-base` 인가

STAGE 1에서 이 모델을 기억하시죠? 정반대 문장에 점수차 **0.0038**을 내던 그 무능한 모델입니다.
분류 헤드가 랜덤 주사위였기 때문이었죠.

**오늘 그 주사위를 진짜 판단기로 훈련시킵니다.** 이야기가 여기서 닫힙니다.

그리고 실용적인 이유도 있습니다 — `klue/roberta-base`는 **한국어 전용**이라,
12만 사전 자리를 104개 언어와 나눠 쓰는 mBERT보다 한국어 성능이 훨씬 좋습니다.
(STAGE 2에서 배운 그 이야기입니다)


In [ ]:
MODEL_NAME  = 'klue/roberta-base'   # 한국어 전용. STAGE 1의 그 모델
# MODEL_NAME = 'bert-base-multilingual-cased'  # 로컬 CPU 실습과 동일 조건으로 비교하고 싶다면

N_SAMPLES   = 20000    # 학습에 쓸 리뷰 수 (NSMC 전체는 15만건)
EPOCHS      = 2
BATCH       = 32       # GPU 메모리가 넉넉하므로 CPU(8)보다 크게
MAX_LENGTH  = 128      # NSMC 리뷰 길이 여유있게
LR          = 3e-5
WARMUP      = 0.1      # 초반 학습률을 서서히 올려 안정화
SEED        = 42

print(f'{MODEL_NAME} / {N_SAMPLES}건 / {EPOCHS}에폭 / batch {BATCH} / len {MAX_LENGTH}')


## 4️⃣ 데이터 로드 + 품질 진단

NSMC = 네이버 영화 리뷰 20만 건. 실제 사람이 쓴 한국어입니다.

`trust_remote_code=True` 가 필요한 이유: NSMC는 옛날 방식(로딩 스크립트)이라
파이썬 코드를 내려받아 실행합니다. datasets 3.x부터 명시적 동의를 요구합니다.


In [ ]:
from datasets import load_dataset

raw = load_dataset('e9t/nsmc', trust_remote_code=True)
print(raw)

train_all = raw['train'].shuffle(seed=SEED)   # ★ 반드시 섞고 나서 자른다
train_sub = train_all.select(range(N_SAMPLES))

texts = [str(t) for t in train_sub['document']]
labels = train_sub['label']

# 어휘 다양성(TTR) — STAGE 2에서 배운 지표
words = [w for t in texts for w in t.split()]
ttr = len(set(words)) / len(words)

print(f"\n문장 수      : {len(texts):,}")
print(f"고유 어절     : {len(set(words)):,}")
print(f"어휘 다양성   : {ttr:.3f}   (템플릿 데이터는 0.023 이었습니다)")
print(f"라벨 분포     : 긍정 {sum(labels):,} / 부정 {len(labels)-sum(labels):,}")
print(f"단어당 노출   : 평균 {len(words)/len(set(words)):.1f}회   (CPU 실습에서는 1.2회였습니다)")
print('\n샘플 3건:')
for t, l in zip(texts[:3], labels[:3]):
    print(f"  [{'긍정' if l else '부정'}] {t[:60]}")


## 5️⃣ 전처리

토큰 길이를 먼저 재서 `max_length` 근거를 만듭니다. STAGE 2에서 배운 습관입니다.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 표본으로 길이 분포 측정
lengths = sorted(len(tokenizer.encode(t)) for t in texts[:2000])
p95 = lengths[int(len(lengths)*0.95)-1]
over = sum(1 for l in lengths if l > MAX_LENGTH)
print(f'토큰 길이: 중앙값 {lengths[len(lengths)//2]} / 95%지점 {p95} / 최대 {lengths[-1]}')
print(f'max_length={MAX_LENGTH} 로 자르면 {over/len(lengths)*100:.1f}% 가 잘립니다')
if over/len(lengths) > 0.05:
    print('⚠️ 5% 넘게 잘립니다. MAX_LENGTH 를 올리는 것을 고려하세요.')
else:
    print('✅ 손실이 적습니다.')

def preprocess(examples):
    return tokenizer(examples['document'], truncation=True,
                     padding='max_length', max_length=MAX_LENGTH)

ds = train_sub.map(preprocess, batched=True)
ds = ds.remove_columns([c for c in ds.column_names
                        if c not in ('input_ids','attention_mask','token_type_ids','label')])

from datasets import ClassLabel
if not isinstance(ds.features['label'], ClassLabel):
    ds = ds.class_encode_column('label')

split = ds.train_test_split(test_size=0.2, seed=SEED, stratify_by_column='label')
split.set_format('torch')
print(split)


## 6️⃣ 학습

`fp16=True` — GPU에서 16비트 연산을 써서 속도를 약 2배 올립니다. CPU에서는 못 쓰는 옵션입니다.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='macro')}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.id2label = {0: '부정', 1: '긍정'}
model.config.label2id = {'부정': 0, '긍정': 1}

args = TrainingArguments(
    output_dir='./results',
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH*2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=WARMUP,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    fp16=True,          # ★ GPU 전용 가속
    report_to=[],
    seed=SEED,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=split['train'], eval_dataset=split['test'],
                  compute_metrics=compute_metrics)

result = trainer.train()
metrics = trainer.evaluate()

import math
print('\n' + '='*60)
print(f"train_loss   : {result.training_loss:.4f}   (무작위 = {math.log(2):.3f})")
print(f"검증 정확도   : {metrics['eval_accuracy']:.1%}")
print(f"검증 F1      : {metrics['eval_f1']:.4f}")
print(f"소요 시간     : {result.metrics['train_runtime']:.0f}초")
print('='*60)


## 7️⃣ 실전 시험 — CPU 모델과 같은 문제

로컬 CPU에서 썼던 **B조 문장 10개**를 그대로 씁니다. 공정한 비교를 위해서요.


In [ ]:
import torch.nn.functional as F

REAL_SENTENCES = [
    ('이건 진짜 내 인생 영화다.. 너무 감동이야.', 1),
    ('보는 내내 눈물이 멈추질 않았어요.', 1),
    ('올해 본 것 중에 제일 좋았습니다.', 1),
    ('두 번 봐도 재밌더라구요.', 1),
    ('친구한테 벌써 세 번 얘기했어요.', 1),
    ('중간에 잠들 뻔했어요. 전개가 너무 느림.', 0),
    ('표값이 아까워서 화가 났습니다.', 0),
    ('도대체 뭘 말하려는 건지 모르겠네요.', 0),
    ('차라리 그 시간에 잠을 잘 걸 그랬어요.', 0),
    ('기대가 컸던 만큼 실망도 컸습니다.', 0),
]

model.eval()
device = next(model.parameters()).device

def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       padding=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0]
    p = int(torch.argmax(probs))
    return p, float(probs[p])

correct = 0
confs_wrong, confs_right = [], []
for text, ans in REAL_SENTENCES:
    p, c = predict(text)
    ok = p == ans
    correct += ok
    (confs_right if ok else confs_wrong).append(c)
    print(f"{'✅' if ok else '❌'} {text}")
    print(f"    정답 {'긍정' if ans else '부정'} / 예측 {'긍정' if p else '부정'} ({c:.1%})")

acc = correct/len(REAL_SENTENCES)
print('\n' + '='*60)
print(f'B조 실전 정답률 : {correct}/{len(REAL_SENTENCES)} = {acc:.0%}')
if confs_wrong:
    print(f'틀렸을 때 평균 확신 : {sum(confs_wrong)/len(confs_wrong):.1%}')
print('='*60)


## 8️⃣ 최종 비교표

세 모델을 나란히 놓고 봅니다.


In [ ]:
import pandas as pd

# 로컬 CPU 실습 결과 (실측값 — 본인 결과가 다르면 여기를 고치세요)
rows = [
    {'모델':'① 템플릿 데이터 (CPU)', '데이터':'600건 템플릿', '어휘':74,
     '학습시간':'239초', '검증정확도':'100%', 'B조 실전':'80%', '틀릴때 확신':'98.4%'},
    {'모델':'② NSMC 1,500건 (CPU)', '데이터':'1,500건 실제', '어휘':8172,
     '학습시간':'477초', '검증정확도':'70%', 'B조 실전':'50%', '틀릴때 확신':'72.3%'},
    {'모델':f'③ NSMC {N_SAMPLES:,}건 (GPU)', '데이터':f'{N_SAMPLES:,}건 실제', '어휘':len(set(words)),
     '학습시간':f"{result.metrics['train_runtime']:.0f}초",
     '검증정확도':f"{metrics['eval_accuracy']:.0%}", 'B조 실전':f'{acc:.0%}',
     '틀릴때 확신':f"{sum(confs_wrong)/len(confs_wrong):.1%}" if confs_wrong else '—'},
]
pd.set_option('display.unicode.east_asian_width', True)
display(pd.DataFrame(rows))


## 9️⃣ 모델 저장 & 내 컴퓨터로 가져오기

학습된 모델을 zip으로 묶어 다운로드합니다.

압축을 풀어 `day3_finetuning/data/colab_model/` 에 넣으면 로컬에서도 쓸 수 있고,
**STAGE 5에서 이 모델을 허깅페이스 Spaces에 배포**하게 됩니다.


In [ ]:
SAVE_DIR = 'nsmc_sentiment_model'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import os
print('저장된 파일:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))/1e6
    print(f'  {f:<32} {size:>7.1f} MB')

!zip -qr {SAVE_DIR}.zip {SAVE_DIR}
print(f'\n✅ {SAVE_DIR}.zip 생성 완료')

from google.colab import files
files.download(f'{SAVE_DIR}.zip')


## 🎓 정리

### 확인해야 할 것

| 항목 | CPU 1,500건 | GPU 20,000건 |
|---|---|---|
| `train_loss` | 0.5952 (무작위 0.693에 근접) | ? |
| 검증 정확도 | 70% | ? |
| B조 실전 | 50% | ? |
| 단어당 노출 | 1.2회 | ? |

### 이번에 확인할 명제

> **다양성은 일반화의 필요조건이지 충분조건이 아니다.**
> 데이터를 어렵게 만들었으면 학습도 그만큼 늘려야 한다.

CPU에서는 '어려운 문제 + 부족한 학습' = 실패였습니다.
GPU에서 학습량을 13배 늘리면 그 명제가 검증됩니다.

### 다음

- 다운로드한 zip을 로컬 `day3_finetuning/data/colab_model/` 에 풀어두세요
- **STAGE 5**에서 이 모델을 Gradio로 감싸 허깅페이스 Spaces에 배포합니다

---

### 💡 Colab 사용 팁

- 무료 GPU는 연속 사용 시간 제한이 있습니다. 학습이 끝나면 **런타임 → 런타임 연결 해제**
- 90분간 조작이 없으면 세션이 끊깁니다. 긴 학습 중에는 가끔 탭을 확인하세요
- 세션이 끊기면 **저장 안 한 모델은 사라집니다.** 학습 직후 바로 다운로드하세요
